# Prime Kernel Homology — TL;DR

A condensed reading of [`prime_kernel_homology.ipynb`](prime_kernel_homology.ipynb).
The full notebook has 60 cells across 10 sections; this version keeps **one
prose paragraph + one minimal code cell per section** so the pipeline is
legible end-to-end.

## What the pipeline does, in four bullets

1. **Verdict matrix.** Read the `lean-kernel-arena/_results/*.json` files
   for 11 kernel checkers × N primes and build an acceptance matrix
   `M ∈ {0,1}^{11×N}`.
2. **Reasoning corpus.** For every cell `(checker, prime)`, narrate the
   proof with the cogitator, regex-tag each step with rellm, and collect
   the tagged steps as a CSV.
3. **Topology.** Embed the steps with sbert → point cloud in ℝ³⁸⁴ →
   Vietoris–Rips → persistent homology. The **3-torus T³ ⊂ ℝ⁶** with
   Betti pattern `(1, 3, 3, 1)` is the reference topology.
4. **Verdict.** Combine `M`, the reasoning Betti table, and the
   1-category consistency theorem `S_rellm_consistency.lean` to score
   each checker.

**Verdict types:** `trust` · `distrust` · `insufficient-data`.


## §0  Imports and paths

In [ ]:
from __future__ import annotations
import itertools, json, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

HOME      = pathlib.Path.home()
ARENA     = HOME / "lean" / "lean-kernel-arena" / "_results"
BUILD_DIR = HOME / "lean" / "_prime_homology_build"
CSV_FILE  = BUILD_DIR / "parsed_steps.csv"

CHECKERS = ["lean4lean", "sonanoda", "nanoda"]
TESTS    = ["prime-2", "prime-3", "prime-5"]


## §1  Arena verdicts → acceptance matrix `M`

For each `(checker, test)`, the JSON has `status ∈ {accepted, declined,
rejected}` and `correctness ∈ {correct, incorrect, ?}`. The acceptance
matrix `M` records `1` exactly when the checker both accepts the proof
**and** the answer is correct.

In [ ]:
rows = []
for ck, ts in itertools.product(CHECKERS, TESTS):
    path = ARENA / f"{ck}_{ts}.json"
    if path.exists():
        r = json.loads(path.read_text())
        rows.append({"checker": ck, "test": ts,
                     "status": r.get("status", "?"),
                     "correctness": r.get("correctness", "?"),
                     "wall_time": float(r.get("wall_time", float("nan")))})
    else:
        rows.append({"checker": ck, "test": ts, "status": "missing",
                     "correctness": "?", "wall_time": np.nan})

df_arena = pd.DataFrame(rows)
M = np.array([[int(r.status == "accepted" and r.correctness == "correct")
               for _, r in df_arena[df_arena.checker == ck].iterrows()]
              for ck in CHECKERS])
print("M =\n", M)
df_arena


## §2–3  Cogitator narration + rellm grammar tagging

The cogitator (an LLM driven by `cogitator.LeastToMost`) decomposes each
proof goal into sub-questions. Each step's free-form answer is passed
through a rellm regex grammar with `gpt2` to tag a `(verb, arg)` pair
from the alphabet `Σ = {reduce, evaluate, unfold, apply, close, ...}`.
The tagged steps are persisted to `parsed_steps.csv` with columns
`checker, test, step, subq, answer, tag, verb, arg`.

**TL;DR uses the cached CSV** — building it costs ~60 s × N_cells of
wall-clock and is opt-in only.

In [ ]:
df_steps = pd.read_csv(CSV_FILE)
print(f"parsed_steps.csv: {len(df_steps)} rows, columns = {list(df_steps.columns)}")
df_steps.head(8)


## §4  Embedding + Vietoris–Rips persistent homology

The sentence-transformer `all-MiniLM-L6-v2` maps each step's answer to a
unit-norm vector in ℝ³⁸⁴. `ripser` then builds the Vietoris–Rips
filtration and returns the persistence diagrams `H₀` (connected
components) and `H₁` (loops). Long bars are robust features; short bars
are noise.

In [ ]:
from sentence_transformers import SentenceTransformer
from ripser import ripser

sbert = SentenceTransformer("all-MiniLM-L6-v2")
X = np.asarray(sbert.encode(df_steps["answer"].tolist(),
                            normalize_embeddings=True), dtype=np.float32)
print("point cloud:", X.shape)

dgms = ripser(X, maxdim=1)["dgms"]
H0, H1 = dgms[0], dgms[1]
print(f"H0 bars: {len(H0)}  (finite: {(np.isfinite(H0[:,1])).sum()})")
print(f"H1 bars: {len(H1)}")


## §5–6  1-skeleton and discrete Morse complex (summary)

At a chosen radius ε, the 1-skeleton is a graph on the point cloud
where `i—j` is an edge iff `d(i, j) ≤ ε`. The discrete-Morse pairing
identifies **critical 0-cells** (one per connected component, i.e. each
distinct proof method) and **critical 1-cells** (one per persistent H₁
loop, i.e. each shared reasoning cycle).

In the full notebook this drives a `networkx` plot. The TL;DR omits the
plot — the same information is the `β₀` and `β₁` of §10 below.

## §7  PID controller over cogitator temperature (summary)

A PID loop targets a fixed number of `decompose` sub-questions per
`(checker, prime)` cell by adjusting the cogitator's softmax temperature.
The trajectory is saved to `pid_trajectory.json`; the **last temperature
value** is what the validator uses when filling missing cells.

The TL;DR does not re-run the controller. See cells 19–22 of the full
notebook for plots and convergence proofs.

## §8  The 3-torus T³ ⊂ ℝ⁶ as reference topology

The reasoning point cloud is compared against a **known** topology: the
3-torus T³ = S¹ × S¹ × S¹ embedded in ℝ⁶, with Betti numbers
`(β₀, β₁, β₂, β₃) = (1, 3, 3, 1)`. If the reasoning complex has a
similar β₁-band, the eleven checkers share three independent reasoning
loops — strong validation.

In [ ]:
# 150 uniform samples from T³ ⊂ ℝ⁶
rng = np.random.default_rng(0)
theta = rng.uniform(0, 2*np.pi, size=(150, 3))
Y = np.column_stack([np.cos(theta[:,0]), np.sin(theta[:,0]),
                     np.cos(theta[:,1]), np.sin(theta[:,1]),
                     np.cos(theta[:,2]), np.sin(theta[:,2])]).astype(np.float32)
torus_dgms = ripser(Y, maxdim=2)["dgms"]
for k, dg in enumerate(torus_dgms):
    print(f"torus H{k}: {len(dg)} bars  (long bars ~ Betti {k})")


## §9  Signal flow as a free-monoid monad

The pipeline factors as a 1-categorical diagram:

```
Σ ──T──► cogitator+rellm steps ──ι(sbert)──► X ⊂ ℝ³⁸⁴ ──VR──► K(ε) ──PH──► β_k
                                                                          │
                                                                       compare
                                                                          ▼
T³ ⊂ ℝ⁶ ──VR──► K_T³(ε) ──PH──► (1,3,3,1)
```

`T` is the free-monoid monad on `Set`; `ι` is sentence-transformer
embedding; the rest is shared. The 1-category consistency of this
diagram is certified by `S_rellm_consistency.lean` — the precondition
for trusting any Betti-based verdict.


In [ ]:
# Emit the .gv source (renders to SVG iff `dot` is installed)
import graphviz as gv
g = gv.Digraph("signal_flow_tldr", engine="dot")
g.attr(rankdir="LR", fontname="Helvetica", fontsize="11")
g.node("sigma", "Σ", shape="parallelogram", style="filled", fillcolor="#FCE4D6")
g.node("T",     "T: X ↦ X*",          shape="box",  style="filled", fillcolor="#F8D7DA")
g.node("steps", "cogitator+rellm",    shape="box",  style="filled", fillcolor="#FFF2CC")
g.node("embed", "ι: sbert",           shape="box",  style="filled", fillcolor="#DAE8FC")
g.node("pc",    "X ⊂ ℝ³⁸⁴",           shape="oval")
g.node("VR",    "Vietoris–Rips",      shape="box",  style="filled", fillcolor="#D5E8D4")
g.node("PH",    "H₀, H₁",             shape="box",  style="filled", fillcolor="#D5E8D4")
g.node("bar",   "barcode",            shape="doublecircle",
       style="filled", fillcolor="#B85450", fontcolor="white")
for a, b in [("sigma","T"), ("T","steps"), ("steps","embed"),
             ("embed","pc"), ("pc","VR"), ("VR","PH"), ("PH","bar")]:
    g.edge(a, b)
g


## §10  Reasoning Betti table and estimator routing

The validator computes `β₀, β₁` (and optionally `β₂`) of the reasoning
clique complex via `betti_qiskit`. The routing rule is:

```
|X| ≤ 64                  → betti_classical (dense)
|X| ≤ 256                 → betti_classical (sparse)
|X| ≤ 1024 or max_dim ≥ 3 → betti_quantum (Aer)
otherwise                 → betti_quantum (IBM hardware)
```

At the current size (`|X| ≈ 122`), classical wins. QPE is held in
reserve for the higher-prime expansion (prime-7, prime-11, prime-13)
where `|S_k|` grows fast in `k`.

In [ ]:
import sys
sys.path.insert(0, str(HOME / "lean"))
import betti_qiskit as bq
from scipy.spatial.distance import squareform, pdist

D = squareform(pdist(X, metric="euclidean"))
eps = float(np.quantile(D[np.triu_indices_from(D, 1)], 0.35))
N = X.shape[0]
edges = [(i, j) for i in range(N) for j in range(i+1, N) if D[i, j] <= eps]
print(f"reasoning graph: |V|={N}, |E|={len(edges)}, eps={eps:.3f}")

K = bq.clique_complex(N, edges, max_dim=2)
betti = {k: bq.betti_classical(K, k) for k in (0, 1, 2)}
print(f"REASONING BETTI:  β₀={betti[0]}  β₁={betti[1]}  β₂={betti[2]}")


## Where the verdict comes from

Combine the three signals:

| Signal | Source | Interpretation |
|--------|--------|----------------|
| `M[c, p]` | `_results/{c}_{p}.json` | did checker `c` accept prime `p` correctly? |
| Reasoning Betti table | this notebook, §10 | does the cell live in the **majority** β₀ component? does it cover a persistent β₁ loop? |
| 1-cat consistency | `S_rellm_consistency.lean` | is the whole functor `PH ∘ VR ∘ ι ∘ T` still well-typed? |

The score formula (see `SKILL.md`, Q6) is a weighted sum; ties break
toward `(checker, prime)` cells whose `verb` tag matches the active
proof family `{reduce, evaluate, unfold, apply, close}`.

## Next steps

- Run the full pipeline end-to-end:
  [`prime_kernel_homology.ipynb`](prime_kernel_homology.ipynb).
- Read the decision-tree spec:
  [`../skills/cogitator-validator/SKILL.md`](../skills/cogitator-validator/SKILL.md).
- Read the implementation plan: [`plan.md`](plan.md).
- Read the LaTeX session record:
  [`implementation-record.pdf`](implementation-record.pdf).
